In [2]:
import numpy as np
import numba as nb
from numba.typed import List
import math
from time import perf_counter
import matplotlib.pyplot as plt
from numba.core.registry import CPUDispatcher

In [3]:
LOG2 = math.log(2.0)


@nb.njit(fastmath=True, cache=True)
def fastlog(x):
    return math.log(x)


@nb.njit(fastmath=False, cache=True)
def v2_numba(r: int) -> int:
    """
    Compute the exponent of the largest power of 2 that divides r.
    """
    if r == 0:
        return -1
    if r < 0:
        r = -r
    c = 0
    while (r & 1) == 0:
        c += 1
        r >>= 1
    return c


@nb.njit(fastmath=False, cache=True)
def get_grid(t):
    """
    Compute the grid Gt for a given time step t. Non-recursive version.

    Parameters
    ----------
    t : int

    Returns
    -------
    np.ndarray (int64)
        The grid Gt as an integer NumPy array.
    """
    # Handle edge case t <= 1 (consistent with the commented R behavior for t==1)
    if t <= 1:
        out = np.empty(1, dtype=np.int64)
        out[0] = 1
        return out

    # Compute upper bounds using base-2 logs via math.log(...)
    # upper1 <- floor(log((t - 1) / 3, base = 2)) + 1
    # upper2 <- floor(log(t - 1, base = 2)) - 1
    upper1 = math.floor(math.log((t - 1) / 3.0) / LOG2) + 1
    upper2 = math.floor(math.log(t - 1) / LOG2) - 1

    # Preallocate output array
    length = 1 + (upper1 if upper1 > 0 else 0) + (upper2 if upper2 > 0 else 0)
    Gt = np.empty(length, dtype=np.int64)

    # Gt[1] <- 1  (R is 1-based; Python is 0-based)
    Gt[0] = 1
    counter = 1

    if upper1 >= 1:
        for j in range(1, upper1 + 1):
            # gL <- 2^j + (t - 1) %% 2^(j - 1)
            gL = (1 << j) + ((t - 1) % (1 << (j - 1)))
            Gt[counter] = gL
            counter += 1

            if j <= upper2:
                # Gt[counter] <- gL + 2^(j - 1)
                Gt[counter] = gL + (1 << (j - 1))
                counter += 1

    return Gt


@nb.njit(fastmath=False, cache=True)
def update_grid_numba(old_grid, t):
    """
    Update the grid old_grid in place to reflect the addition of a new data point at time t.
    Parameters
    ----------
    old_grid : List[int]
        The current grid to be updated.
    t : int
        The current (i.e., new) time step.
    """
    removed = -1
    removed_g = -1

    if t == 2 or t == 3:
        removed_g = old_grid[0]
        old_grid.pop(0)
        removed = 0
    elif t > 3:
        j = v2_numba(t - 1) + 1
        if j > 0:
            ind = 2 * j

            if ind < len(old_grid):
                removed = len(old_grid) - ind - 1
                removed_g = old_grid[removed]
                old_grid.pop(removed)

    old_grid.append(-t)

    return removed, removed_g


def is_numba_func(f):
    return isinstance(f, CPUDispatcher)


@nb.njit
def init_state_numba(v):
    grid_list = List.empty_list(nb.int64)
    sum_pre_list = List.empty_list(np.zeros(v, dtype=np.float64))
    return grid_list, sum_pre_list


def init_state(p, h, f, penalty, penalty_constant, auxiliary_data=None):
    # Determine v by probing h on a dummy input
    dummy_y = np.zeros(p, dtype=np.float64)
    h_y = h(dummy_y)
    v = h_y.shape

    use_numba = is_numba_func(h) and is_numba_func(f) and is_numba_func(penalty)

    state = {
        "t": 0,
        "p": p,
        "v": v,
        "use_numba": use_numba,
        "h": h,
        "f": f,
        "penalty": penalty,
        "penalty_constant": penalty_constant,
        "auxiliary_data": auxiliary_data,
    }

    if use_numba:
        # Create Numba state
        grid_list, sum_pre_list = init_state_numba(v)
        state["grid_list"] = grid_list
        state["sum_pre_list"] = sum_pre_list
        state["sum"] = np.zeros(v, dtype=np.float64)  # Initialize sum as a NumPy array
        state["alarm"] = False
        state["maxx"] = 0.0
        state["maxpos"] = -1

    else:
        # Create Python/NumPy equivalents (lists/arrays)
        state["t"] = 0
        state["grid_list"] = List.empty_list(nb.int64)
        state["sum_pre_list"] = []
        state["sum"] = np.zeros(v, dtype=np.float64)  # Initialize sum as a NumPy array
        state["alarm"] = False
        state["maxx"] = 0.0
        state["maxpos"] = -1

    return state


@nb.njit(fastmath=False, cache=True)
def update_data_grid_numba(x_new, old_sums, old_S, removed, h):
    if removed >= 0:
        old_sums.pop(removed)
    old_sums.append(old_S)
    tmp = h(x_new)
    S = old_S + tmp
    return S


def update_data_grid(x_new, old_sums, old_S, removed, h):
    if removed >= 0:
        old_sums.pop(removed)
    old_sums.append(old_S)
    tmp = h(x_new)
    S = old_S + tmp
    return S


@nb.njit(fastmath=False, cache=True)
def update_numba(
    x_new,
    p,
    t,
    grid_list,
    sum_pre_list,
    S,
    maxx,
    maxpos,
    alarm,
    h,
    f,
    penalty,
    penalty_constant,
):
    removed, removed_g = update_grid_numba(grid_list, t)
    S_new = update_data_grid_numba(x_new, sum_pre_list, S, removed, h)

    if t > 1:
        for j in range(len(grid_list) - 1, -1, -1):
            g = grid_list[j] + t + 1
            val = f(sum_pre_list[j], S_new - sum_pre_list[j], g, t)
            pen = penalty(g, t, p)
            cc = val / pen
            if cc > maxx:
                maxx = cc
                maxpos = grid_list[j] + t + 1
            if cc > penalty_constant:
                alarm = True

    return grid_list, sum_pre_list, S_new, maxx, maxpos, alarm


def update_python(x_new, state):
    removed, removed_g = update_grid_numba(state["grid_list"], state["t"])
    state["sum"] = update_data_grid(
        x_new, state["sum_pre_list"], state["sum"], removed, state["h"]
    )

    if state["t"] > 1:
        for j in range(len(state["grid_list"]) - 1, -1, -1):
            g = state["grid_list"][j] + state["t"] + 1
            val = state["f"](
                state["sum_pre_list"][j],
                state["sum"] - state["sum_pre_list"][j],
                g,
                state["t"],
            )
            pen = state["penalty"](g, state["t"], state["p"])
            cc = val / pen
            if cc > state["maxx"]:
                state["maxx"] = cc
                state["maxpos"] = state["grid_list"][j] + state["t"] + 1
            if cc > state["penalty_constant"]:
                state["alarm"] = True

    return state


def update(x_new, state):
    state["t"] = state["t"] + 1
    if state["use_numba"]:
        grid_list, sum_pre_list, S, maxx, maxpos, alarm = update_numba(
            x_new,
            state["p"],
            state["t"],
            state["grid_list"],
            state["sum_pre_list"],
            state["sum"],
            state["maxx"],
            state["maxpos"],
            state["alarm"],
            state["h"],
            state["f"],
            state["penalty"],
            state["penalty_constant"],
        )
        state["grid_list"] = grid_list
        state["sum_pre_list"] = sum_pre_list
        state["sum"] = S
        state["maxx"] = maxx
        state["maxpos"] = maxpos
        state["alarm"] = alarm
    else:
        state = update_python(x_new, state)


@nb.njit(fastmath=False, cache=True)
def mc_max_statistics_numba_driver(X, p, v, penalty_constant, h, f, penalty):
    """
    Numba driver for Monte Carlo max statistic computation.

    Parameters
    ----------
    X : np.ndarray, shape (K, N, p), float64
        Pre-generated null data.
    p : int
        Dimension of data.
    v : tuple(int, ...)
        Shape of h(x) output (same as state["v"]).
    penalty_constant : float
        As in your online algorithm.
    h, f, penalty : numba compiled callables.

    Returns
    -------
    max_values : np.ndarray, shape (K,)
        Max statistic for each MC sample.
    """
    K, N, _ = X.shape
    max_values = np.empty(K, dtype=np.float64)

    for k in range(K):
        # Initialize state pieces for this stream
        grid_list = List.empty_list(nb.int64)
        sum_pre_list = List.empty_list(np.zeros(v, dtype=np.float64))
        S = np.zeros(v, dtype=np.float64)
        maxx = 0.0
        maxpos = -1
        alarm = False

        # Process stream
        for t in range(N):
            x_new = X[k, t, :]
            grid_list, sum_pre_list, S, maxx, maxpos, alarm = update_numba(
                x_new,
                p,
                t + 1,  # time starts at 1 in your update
                grid_list,
                sum_pre_list,
                S,
                maxx,
                maxpos,
                alarm,
                h,
                f,
                penalty,
                penalty_constant,
            )

        max_values[k] = maxx

    return max_values


def mc_max_statistics_python(
    X,
    p,
    h,
    f,
    penalty,
    penalty_constant=0.0,
    auxiliary_data=None,
):
    """
    Pure-Python MC loop, given pre-generated data X.

    Parameters
    ----------
    X : np.ndarray, shape (K, N, p)
        Pre-generated null data.
    p : int
        Dimension of data.
    h, f, penalty : callables
    penalty_constant : float, optional
    auxiliary_data : any, optional

    Returns
    -------
    max_values : np.ndarray, shape (K,)
    """
    K, N, _ = X.shape
    max_values = np.empty(K, dtype=np.float64)

    for k in range(K):
        # Init state for this run (Python path because we call this only when use_numba=False)
        state = init_state(
            p=p,
            h=h,
            f=f,
            penalty=penalty,
            penalty_constant=penalty_constant,
            auxiliary_data=auxiliary_data,
        )

        for t in range(N):
            x_new = X[k, t, :]
            update(x_new, state)  # will go through update_python

        max_values[k] = state["maxx"]

    return max_values


def mc_max_statistics(
    N,
    K,
    p,
    h,
    f,
    penalty,
    null_dist,
    penalty_constant=0.0,
    null_args=(),
    null_kwargs=None,
    auxiliary_data=None,
):
    """
    Wrapper to compute MC max statistics, choosing Python or Numba path.

    Parameters
    ----------
    N : int
        Length of each data stream.
    K : int
        Number of Monte Carlo samples.
    p : int
        Dimension of the data x_t.
    h, f, penalty : callables
        As in your online algorithm.
    null_dist : callable
        Random generator, called as:
            null_dist(*null_args, size=(K, N, p), **null_kwargs)
    penalty_constant : float, optional
    null_args : tuple, optional
    null_kwargs : dict, optional
    auxiliary_data : any, optional

    Returns
    -------
    max_values : np.ndarray, shape (K,)
        Max statistic per MC sample.
    """
    if null_kwargs is None:
        null_kwargs = {}

    # Probe h to determine v (shape of h(x))
    dummy_y = np.zeros(p, dtype=np.float64)
    h_y = h(dummy_y)
    v = h_y.shape

    use_numba = is_numba_func(h) and is_numba_func(f) and is_numba_func(penalty)

    # Generate all null data once
    X = null_dist(*null_args, size=(K, N, p), **null_kwargs).astype(np.float64)

    if not use_numba:
        # Pure Python path: pass X to the Python driver
        return mc_max_statistics_python(
            X=X,
            p=p,
            h=h,
            f=f,
            penalty=penalty,
            penalty_constant=penalty_constant,
            auxiliary_data=auxiliary_data,
        )
    else:
        # Numba path: pass X to the Numba driver (no Python calls in the inner loop)
        return mc_max_statistics_numba_driver(
            X=X,
            p=p,
            v=v,
            penalty_constant=penalty_constant,
            h=h,
            f=f,
            penalty=penalty,
        )

### Univariate change-in-mean

### Univariate Gaussian CUSUM example

In [3]:
@nb.njit
def h(y):
    return y


@nb.njit
def f(sum_pre_j, sum_post_j, g, t):
    res = math.sqrt(1.0 * g / (t * (t - g))) * sum_pre_j
    res = res - math.sqrt(1.0 * (t - g) / t / g) * sum_post_j
    return np.sum(res * res)


@nb.njit
def penalty(g, t, p):
    logg = fastlog(t / 0.05)
    logg = logg + math.sqrt(logg)
    return logg

In [4]:
N = 1000
K = 1000
p = 1
null_dist = np.random.normal
null_args = (0, 1)  # mean=0, std=1
sample = mc_max_statistics(
    N=N,
    K=K,
    p=p,
    h=h,
    f=f,
    penalty=penalty,
    null_dist=null_dist,
    penalty_constant=0.0,
    null_args=null_args,
    null_kwargs=None,
    auxiliary_data=None,
)

In [5]:
critical_value = np.percentile(sample, 95)
print(f"Critical value at 95%: {critical_value}")

Critical value at 95%: 1.75911643521425


In [6]:
N = 1000
xs = np.random.normal(0, 1, (N, p))
xs[(N // 2) :] += 0.5  # Introduce a change point at iteration 8000

state = init_state(p, h, f, penalty, critical_value)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

Alarm triggered at iteration 595 with maxx = 1.7701010511505497 at position 82


### Multivariate Gaussian CUSUM example (dense change)

In [7]:
@nb.njit
def h(y):
    return y


@nb.njit
def f(sum_pre_j, sum_post_j, g, t):
    res = math.sqrt(1.0 * g / (t * (t - g))) * sum_pre_j
    res = res - math.sqrt(1.0 * (t - g) / t / g) * sum_post_j
    return np.sum(res * res)


@nb.njit
def penalty(g, t, p):
    logg = fastlog(t / 0.05)
    rr = math.sqrt(p * logg) + logg

    return rr

In [8]:
N = 1000
K = 1000
p = 10
null_dist = np.random.normal
null_args = (0, 1)  # mean=0, std=1
sample = mc_max_statistics(
    N=N,
    K=K,
    p=p,
    h=h,
    f=f,
    penalty=penalty,
    null_dist=null_dist,
    penalty_constant=0.0,
    null_args=null_args,
    null_kwargs=None,
    auxiliary_data=None,
)

In [9]:
critical_value = np.percentile(sample, 95)
print(f"Critical value at 95%: {critical_value}")

Critical value at 95%: 2.5269117232949907


In [10]:
xs = np.random.normal(0, 1, (N, p))
xs[(N // 2) :] += 0.2  # Introduce a change point at iteration 8000

state = init_state(p, h, f, penalty, critical_value)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

Alarm triggered at iteration 646 with maxx = 2.562655347811028 at position 133


### Multivariate change in mean or covariance

In [ ]:
@nb.njit
def h_mean_cov(y):
    """
    Sufficient statistic for Gaussian mean+covariance:
    Return concatenation of y and vec(y y^T).

    If y has shape (p,), this returns an array of shape (p + p*p,)
    where the last p*p entries are column-major flattening of y y^T.
    """
    p = y.shape[0]
    yy = np.outer(y, y)
    out = np.empty((p + 1, p), dtype=y.dtype)
    out[0] = y
    out[1:] = yy

    return out


@nb.njit
def f_mean_cov(sum_pre_j, sum_post_j, g, t):
    """
    GLR-type statistic for a change in both mean and covariance
    in multivariate Gaussian data.

    sum_pre_j: sum of h(y) over segment 1  (shape (p+1, p))
    sum_post_j: sum of h(y) over segment 2 (shape (p+1, p))
    g: candidate change-point (segment 1 length)
    t: total sample size
    """
    n1 = t - g
    n2 = g

    p = sum_pre_j.shape[0]  # dimension of data
    if n1 <= p or n2 <= p:
        return 0.0

    totalsum = sum_pre_j + sum_post_j
    sum_pre_j_id = sum_pre_j[0]
    sum_post_j_id = sum_post_j[0]
    totalsum_id = totalsum[0]

    sum_pre_j_cov = sum_pre_j[1:]
    sum_post_j_cov = sum_post_j[1:]
    totalsum_cov = totalsum[1:]

    Sigma_tot = (totalsum_cov - np.outer(totalsum_id, totalsum_id) / t) / t
    Sigma_pre_j = (sum_pre_j_cov - np.outer(sum_pre_j_id, sum_pre_j_id) / n1) / n1
    Sigma_post_j = (sum_post_j_cov - np.outer(sum_post_j_id, sum_post_j_id) / n2) / n2

    # GLR statistic: t * log|Sigma| - g* log|Sigma1| - (t-g) * log|Sigma2|
    # Use slogdet for numerical stability
    sign0, logdet0 = np.linalg.slogdet(Sigma_tot)
    sign1, logdet1 = np.linalg.slogdet(Sigma_pre_j)
    sign2, logdet2 = np.linalg.slogdet(Sigma_post_j)

    if False:
        # If any covariance is singular (sign <= 0), treat statistic as 0
        if sign0 <= 0 or sign1 <= 0 or sign2 <= 0:
            if sign0 <= 0:
                print("Sigma_tot is singular or not positive definite")
            if sign1 <= 0:
                print("Sigma_pre_j is singular or not positive definite")
            if sign2 <= 0:
                print("Sigma_post_j is singular or not positive definite")
            return 0.0

    LR = t * logdet0 - n1 * logdet1 - n2 * logdet2
    df = p + (p * (p + 1)) // 2  # Number of parameters in mean+covariance

    return LR - df


@nb.njit
def penalty_mean_cov(g, t, p):
    df = (p * (p + 1)) // 2 + p
    logg = fastlog(t / 0.05)
    rr = math.sqrt(df * logg) + logg

    return rr

In [12]:
p = 5
S1 = np.zeros((p + 1, p))
S2 = np.zeros((p + 1, p))

T1 = np.zeros((p + 1, p))
T2 = np.zeros((p + 1, p))

NN = 100
for i in range(NN):
    y = np.random.normal(0, 1, p)
    h_y = h_mean_cov(y)
    S1 += h_y
    hy = h_mean_cov(y / 2)
    T1 += hy

    y = np.random.normal(0, 1, p)
    h_y = h_mean_cov(y)
    S2 += h_y
    hy = h_mean_cov(y / 2)
    T2 += hy


r1 = f_mean_cov(S1, S2, 50, 100)
r2 = f_mean_cov(T1, T2, 50, 100)
print(r1)
print(r2)

-18.211165434149734
-18.211165434149734


In [13]:
N = 1000
K = 1000
p = 10
null_dist = np.random.normal
null_args = (0, 1)  # mean=0, std=1
sample = mc_max_statistics(
    N=N,
    K=K,
    p=p,
    h=h_mean_cov,
    f=f_mean_cov,
    penalty=penalty_mean_cov,
    null_dist=null_dist,
    penalty_constant=0.0,
    null_args=null_args,
    null_kwargs=None,
    auxiliary_data=None,
)

In [14]:
critical_value = np.percentile(sample, 95)
print(f"Critical value at 95%: {critical_value}")

Critical value at 95%: 4.895632822137039


In [32]:
xs = np.random.normal(0, 1, (N, p))
xs[(N // 2) :] += 0.4

state = init_state(p, h_mean_cov, f_mean_cov, penalty_mean_cov, critical_value)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

Alarm triggered at iteration 666 with maxx = 4.926794872606431 at position 153


In [ ]:
xs = np.zeros((N, p))
xs[: (N // 2)] = np.random.normal(0, 1, (N // 2, p))
xs[(N // 2) :] = np.random.normal(0, 0.5, (N // 2, p))  # Introduce a changepoint


state = init_state(p, h_mean_cov, f_mean_cov, penalty_mean_cov, critical_value)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

Alarm triggered at iteration 535 with maxx = 4.919645782353827 at position 38


In [ ]:
state

### General likelihood ratio test using Numba, requiring gradients

In [ ]:
## Specific examples are for Bernoulli with natural param theta = log(p/(1-p)) and A(theta) = log(1 + exp(theta))


from matplotlib.pylab import inf


@nb.njit
def h(y):
    return y


@nb.njit
def A_func(theta):
    return np.log(1.0 + np.exp(theta)).sum()


@nb.njit
def grad_A(theta):
    ee = np.exp(theta)
    return ee / (1.0 + ee)


@nb.njit
def hess_A(theta):
    H = np.zeros((1, 1), dtype=np.float64)
    ee = np.exp(theta)
    H[0, 0] = (ee / ((1.0 + ee) ** 2)).sum()
    return H


@nb.njit
def penalty(g, t, p):
    logg = fastlog(t / 0.05)
    rr = math.sqrt(p * logg) + logg

    return rr


## The below should be general


@nb.njit
def logLik(theta, S, n):
    # S is cumulative sum!
    # n is total sample size

    # if S is numerically zero or n, we return zero:
    if np.all(np.abs(S) < 1e-10) or np.all(np.abs(S - n) < 1e-10):
        return 0.0
    A_theta = A_func(theta)
    return np.dot(theta, S) - n * A_theta


@nb.njit
def newton_mle(
    S, n, theta_init, A_func, grad_A, hess_A, logLik, tol=1e-12, max_iter=1000
):
    """
    Maximize logLik(theta; S, n) = theta @ S - n * A_func(theta)
    over theta ∈ R^v using safeguarded Newton.

    Inputs
    ------
    S : 1D array (v,)
        Sum of sufficient statistics.
    n : int
        Number of observations.
    theta_init : 1D array (v,)
        Starting point (e.g. previous MLE or some fixed value).
    tol : float
        Stop when ||grad||_inf < tol.
    max_iter : int
        Hard cap on number of iterations.

    Returns
    -------
    theta_hat : 1D array (v,)
    converged : boolean
    """

    if (np.abs(S) < 1e-10).all():
        return np.array([-np.inf]), True

    if (np.abs(S - n) < 1e-10).all():
        return np.array([np.inf]), True

    theta = theta_init
    v = theta.shape[0]

    # Small ridge for numerical stability of Hessian
    ridge = 1e-8

    # current log-likelihood
    ll_old = logLik(theta, S, n)

    for it in range(max_iter):
        # Gradient of log-likelihood: S - n * grad_A(theta)
        gA = grad_A(theta)  # shape (v,)
        grad = S - n * gA  # shape (v,)

        # Check convergence: ||grad||_inf < tol
        max_abs_grad = 0.0
        for i in range(v):
            val = grad[i]
            if val < 0.0:
                val = -val
            if val > max_abs_grad:
                max_abs_grad = val
        if max_abs_grad < tol:
            return theta, True

        # Hessian of log-likelihood: H = -n * hess_A(theta)
        HA = hess_A(theta)  # shape (v, v)
        H = np.empty_like(HA)
        for i in range(v):
            for j in range(v):
                H[i, j] = -n * HA[i, j]
        # Add a small ridge to diagonal: H ← H − ridge * I
        # (H is negative semidefinite; we move it slightly more negative
        # to make it better-conditioned)
        for i in range(v):
            H[i, i] = H[i, i] - ridge

        # Solve H * step = grad
        # (instead of explicitly inverting H)
        try:
            step = np.linalg.solve(H, grad)
        except Exception:
            # If solve fails for some reason, abort
            return theta, False

        # Safeguard: start with full Newton step, shrink if ll decreases
        step_scale = 1.0
        theta_new = theta.copy()
        ll_new = ll_old

        # At most a few shrink steps (hard-coded; no user tuning)
        for _ in range(6):
            for i in range(v):
                theta_new[i] = theta[i] + step_scale * step[i]

            ll_candidate = logLik(theta_new, S, n)

            if np.isfinite(ll_candidate) and ll_candidate >= ll_old:
                ll_new = ll_candidate
                break  # accept this step
            else:
                step_scale *= 0.5  # shrink and try again

        # Update
        theta = theta_new
        ll_old = ll_new

    # If we fall out of the loop, we hit max_iter
    return theta, False


@nb.njit
def f(sum_pre_j, sum_post_j, g, t):
    print("###################")
    print("Iter ", t, " g ", g)
    theta_init = np.zeros(sum_pre_j.shape[0], dtype=np.float64)
    theta0, converged = newton_mle(
        sum_pre_j + sum_post_j, t, theta_init, A_func, grad_A, hess_A, logLik
    )
    print("theta0 ", theta0)
    print("converged = ", converged)
    theta1, converged1 = newton_mle(
        sum_pre_j, t - g, theta0, A_func, grad_A, hess_A, logLik
    )
    theta2, converged2 = newton_mle(
        sum_post_j, g, theta0, A_func, grad_A, hess_A, logLik
    )
    print("theta1 ", theta1)
    print("converged1 = ", converged1)
    print("theta2 ", theta2)
    print("converged2 = ", converged2)
    ll0 = logLik(theta0, sum_pre_j + sum_post_j, t)
    ll1 = logLik(theta1, sum_pre_j, t - g)
    ll2 = logLik(theta2, sum_post_j, g)
    print("ll0 = ", ll0)
    print("ll1 = ", ll1)
    print("ll2 = ", ll2)
    print("LR = ", ll1 + ll2 - ll0)
    return ll1 + ll2 - ll0

In [ ]:
p = 1
penalty_constant = 3.0
N = 10
xs = np.zeros(N)
p0 = 0.1
p1 = 0.8
xs[: (N // 2)] = np.random.binomial(n=1, p=p0, size=N // 2)
xs[(N // 2) :] = np.random.binomial(n=1, p=p1, size=N // 2)

xs[:10]

In [ ]:
state = init_state(p, h, f, penalty, penalty_constant)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

### General likelihood ratio test (non-Numba) using autograd. Fully automatic. sucks

In [ ]:
import autograd.numpy as anp
from autograd import grad
from scipy.optimize import minimize

## Currently, the binomial:


def h(y):
    return y


def A_func(theta):
    ## theta is natural parameter, for bernoulli, A(theta) = log(1 + exp(theta))
    ## and theta = log(p/(1-p)) where p is the Bernoulli paramete
    ret = anp.log(1.0 + anp.exp(theta))
    return ret


def logLik(theta, S, n):
    # S is cumulative sum!
    A_theta = A_func(theta)
    return anp.dot(theta, S) - n * A_theta


theta_init = anp.zeros(1)
df = 1
p = 1


def f(sum_pre_j, sum_post_j, g, t):
    # compute MLEs
    # Overall MLE:
    def objective(theta_flat):
        return -logLik(
            theta_flat, sum_pre_j + sum_post_j, t
        )  # negative for minimization

    def objective_grad(theta_flat):
        theta = theta_flat
        gg = grad(logLik, argnum=0)(theta, sum_pre_j + sum_post_j, t)  # shape (v,)
        return -gg  # because we minimize the negative

    res = minimize(
        objective,
        x0=theta_init,
        jac=objective_grad,
        method="L-BFGS-B",
        options={"maxiter": 200, "disp": False},
    )
    theta0 = res.x

    def objective(theta_flat):
        return -logLik(theta_flat, sum_pre_j, t - g)  # negative for minimization

    def objective_grad(theta_flat):
        theta = theta_flat
        gg = grad(logLik, argnum=0)(theta, sum_pre_j, t - g)  # shape (v,)
        return -gg  # because we minimize the negative

    res = minimize(
        objective,
        x0=theta0,
        jac=objective_grad,
        method="L-BFGS-B",
        options={"maxiter": 200, "disp": False},
    )
    theta1 = res.x

    def objective(theta_flat):
        return -logLik(theta_flat, sum_post_j, g)  # negative for minimization

    def objective_grad(theta_flat):
        theta = theta_flat
        gg = grad(logLik, argnum=0)(theta, sum_post_j, g)  # shape (v,)
        return -gg  # because we minimize the negative

    res = minimize(
        objective,
        x0=theta0,
        jac=objective_grad,
        method="L-BFGS-B",
        options={"maxiter": 200, "disp": False},
    )
    theta2 = res.x

    ret = (
        logLik(theta0, sum_pre_j + sum_post_j, t)
        + logLik(theta1, sum_pre_j, t - g)
        + logLik(theta2, sum_post_j, g)
        - df
    )
    print("ret = ", ret)
    return ret


@nb.njit
def penalty(g, t, p):
    logg = fastlog(t / 0.05)
    rr = math.sqrt(p * logg) + logg

    return rr

In [ ]:
p = 1
penalty_constant = 3.0
N = 100
xs = np.zeros(N)
p0 = 0.5
p1 = 0.8
xs[: (N // 2)] = np.random.binomial(n=1, p=p0, size=N // 2)
xs[(N // 2) :] = np.random.binomial(n=1, p=p1, size=N // 2)

state = init_state(p, h, f, penalty, penalty_constant)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

In [ ]:
state